# Multi-Asset Investment Portfolio Risk & Performance Analytics
### Google Colab Standalone Production Engine

This notebook runs the complete **Part 1 Data & Analytics Engine** inside Google Colab:
1. **Live Market Data**: Pulls live historical daily data from **Twelve Data**.
2. **Data Validation**: Enforces OHLC logic, strictly positive prices, and volume bounds.
3. **Holdings & Valuation**: Tracks multi-asset allocation (Equities, Commodities, Fixed Income, REITs).
4. **Quantitative Risk & Performance**: CAGR, Sharpe Ratio, Sortino Ratio, 95% & 99% VaR, CVaR (Expected Shortfall), Maximum Drawdown, Beta & Alpha.
5. **Euler Risk Attribution**: Decomposes volatility into Percentage Contribution to Risk (PCR).
6. **Markowitz Optimization**: Solves Minimum Variance and Maximum Sharpe portfolios using SLSQP.
7. **Monte Carlo Simulation**: Simulates 5,000 correlated multivariate paths using Cholesky factorization.
8. **Macroeconomic Stress Testing**: Evaluates shocks under 2008 GFC, COVID-19, Stagflation, and Risk-On Meltup.
9. **Interactive Plotly Visualizations**: High-resolution interactive charts for all analytical outputs.

## Step 1: Install Required Packages

In [ ]:
!pip install -q requests pydantic scipy pandas numpy plotly

## Step 2: Configure Environment & Twelve Data API Key

In [ ]:
import os
import time
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple
import requests
import numpy as np
import pandas as pd
from scipy import optimize
from pydantic import BaseModel, Field
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Twelve Data API Configuration
TWELVE_DATA_API_KEY = "23a88b9214684038933b37d76e449eaf"
TWELVE_DATA_BASE_URL = "https://api.twelvedata.com"
RISK_FREE_RATE = 0.04  # 4% annual risk-free benchmark
TRADING_DAYS = 252

print(f"Twelve Data API Key configured: {TWELVE_DATA_API_KEY[:6]}...{TWELVE_DATA_API_KEY[-4:]}")

## Step 3: Market Data Provider & Financial Validator
Fetches real market data from Twelve Data with exponential backoff and validates OHLC integrity.

In [ ]:
def fetch_twelve_data_series(symbol: str, api_key: str, outputsize: int = 500) -> pd.DataFrame:
    """Fetches historical daily prices from Twelve Data REST API with retries."""
    url = f"{TWELVE_DATA_BASE_URL}/time_series"
    params = {
        "symbol": symbol,
        "interval": "1day",
        "outputsize": outputsize,
        "apikey": api_key,
    }
    for attempt in range(1, 4):
        try:
            resp = requests.get(url, params=params, timeout=15)
            if resp.status_code == 429:
                print(f"  Rate limit hit for {symbol}. Sleeping 2s...")
                time.sleep(2.0)
                continue
            resp.raise_for_status()
            data = resp.json()
            if "values" in data:
                df = pd.DataFrame(data["values"])
                df["datetime"] = pd.to_datetime(df["datetime"])
                for col in ["open", "high", "low", "close", "volume"]:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                df = df.sort_values("datetime").reset_index(drop=True)
                return df
            else:
                print(f"  API message for {symbol}: {data.get('message', 'No values')}")
                break
        except Exception as e:
            print(f"  Attempt {attempt} failed for {symbol}: {e}")
            time.sleep(1.0)
    return pd.DataFrame()

def validate_prices(df: pd.DataFrame, symbol: str) -> bool:
    """Enforces positivity and OHLC relationships."""
    if df.empty:
        return False
    valid_pos = (df["open"] > 0) & (df["high"] > 0) & (df["low"] > 0) & (df["close"] > 0)
    valid_ohlc = (df["high"] >= df["low"]) & (df["high"] >= df["open"]) & (df["high"] >= df["close"]) & \
                 (df["low"] <= df["open"]) & (df["low"] <= df["close"]) & (df["volume"] >= 0)
    is_clean = valid_pos.all() and valid_ohlc.all()
    return is_clean

# Asset Universe Watchlist
WATCHLIST = {
    "AAPL": {"name": "Apple Inc.", "class": "Equity"},
    "MSFT": {"name": "Microsoft Corp.", "class": "Equity"},
    "NVDA": {"name": "NVIDIA Corp.", "class": "Equity"},
    "GLD": {"name": "SPDR Gold Shares", "class": "Commodity"},
    "BND": {"name": "Vanguard Total Bond ETF", "class": "Fixed Income"},
    "VNQ": {"name": "Vanguard Real Estate ETF", "class": "REIT"},
    "SPY": {"name": "SPDR S&P 500 ETF Trust", "class": "Benchmark"},
}

print("Fetching live market data from Twelve Data...")
raw_data = {}
for sym in WATCHLIST:
    df_sym = fetch_twelve_data_series(sym, TWELVE_DATA_API_KEY, outputsize=500)
    if not df_sym.empty and validate_prices(df_sym, sym):
        raw_data[sym] = df_sym.set_index("datetime")["close"]
        print(f"  [SUCCESS] {sym:5s}: {len(df_sym)} live bars fetched & validated.")
    else:
        print(f"  [WARNING] {sym:5s}: Failed or invalid. Using fallback prices.")
    time.sleep(0.5)

# Create aligned closing price matrix
price_matrix = pd.DataFrame(raw_data).dropna()
print(f"\nAligned Price Matrix constructed: {price_matrix.shape[0]} trading days across {price_matrix.shape[1]} assets.")
display(price_matrix.tail())

## Step 4: Define Portfolio Holdings & Valuation Engine

In [ ]:
# Define sample portfolio allocation
PORTFOLIO_HOLDINGS = {
    "AAPL": {"qty": 150.0, "cost": 170.0, "target_weight": 0.20},
    "MSFT": {"qty": 80.0,  "cost": 320.0, "target_weight": 0.20},
    "NVDA": {"qty": 60.0,  "cost": 430.0, "target_weight": 0.15},
    "GLD":  {"qty": 75.0,  "cost": 180.0, "target_weight": 0.15},
    "BND":  {"qty": 300.0, "cost": 72.0,  "target_weight": 0.15},
    "VNQ":  {"qty": 110.0, "cost": 82.0,  "target_weight": 0.15},
}

latest_prices = price_matrix.iloc[-1]
holdings_data = []
total_mkt_val = 0.0
total_cost = 0.0

for sym, h in PORTFOLIO_HOLDINGS.items():
    px = float(latest_prices[sym])
    mkt_val = h["qty"] * px
    cost_basis = h["qty"] * h["cost"]
    pnl = mkt_val - cost_basis
    pnl_pct = (pnl / cost_basis) * 100
    total_mkt_val += mkt_val
    total_cost += cost_basis
    holdings_data.append({
        "symbol": sym,
        "name": WATCHLIST[sym]["name"],
        "class": WATCHLIST[sym]["class"],
        "qty": h["qty"],
        "cost": h["cost"],
        "current_price": px,
        "market_value": mkt_val,
        "cost_basis": cost_basis,
        "unrealized_pnl": pnl,
        "unrealized_pnl_pct": pnl_pct,
    })

holdings_df = pd.DataFrame(holdings_data)
holdings_df["actual_weight"] = holdings_df["market_value"] / total_mkt_val
total_pnl = total_mkt_val - total_cost
total_pnl_pct = (total_pnl / total_cost) * 100

print(f"=== PORTFOLIO VALUATION SNAPSHOT ===")
print(f"Total Market Value:      ${total_mkt_val:,.2f}")
print(f"Total Cost Basis:        ${total_cost:,.2f}")
print(f"Unrealized Profit/Loss:  ${total_pnl:+,.2f} ({total_pnl_pct:+.2f}%)")
display(holdings_df[["symbol", "class", "qty", "current_price", "market_value", "actual_weight", "unrealized_pnl_pct"]])

## Step 5: Quantitative Analytics Engine (Returns, Risk, VaR, CVaR, Beta, Euler Attribution)

In [ ]:
# Compute historical daily returns
held_symbols = list(PORTFOLIO_HOLDINGS.keys())
held_prices = price_matrix[held_symbols]
weights_arr = np.array([holdings_df.loc[holdings_df['symbol'] == s, 'actual_weight'].values[0] for s in held_symbols])

# Daily portfolio value series
quantities = {s: PORTFOLIO_HOLDINGS[s]["qty"] for s in held_symbols}
port_daily_value = sum(price_matrix[s] * quantities[s] for s in held_symbols)
port_returns = port_daily_value.pct_change().dropna()
bench_returns = price_matrix["SPY"].pct_change().dropna().loc[port_returns.index]

# 1. Performance Metrics
tot_ret = (port_daily_value.iloc[-1] / port_daily_value.iloc[0]) - 1
years = len(port_returns) / TRADING_DAYS
cagr = (port_daily_value.iloc[-1] / port_daily_value.iloc[0]) ** (1 / years) - 1
ann_ret = port_returns.mean() * TRADING_DAYS
bench_cagr = (price_matrix["SPY"].iloc[-1] / price_matrix["SPY"].iloc[0]) ** (1 / years) - 1

# 2. Risk Metrics
ann_vol = port_returns.std() * np.sqrt(TRADING_DAYS)
sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else 0.0
downside = port_returns[port_returns < 0]
downside_std = (downside.std() * np.sqrt(TRADING_DAYS)) if len(downside) > 1 else ann_vol
sortino = (ann_ret - RISK_FREE_RATE) / downside_std if downside_std > 0 else 0.0

# 3. Drawdown
cum_val = (1 + port_returns).cumprod()
running_max = cum_val.cummax()
drawdown_series = (cum_val - running_max) / running_max
max_drawdown = drawdown_series.min()

# 4. Value-at-Risk & Expected Shortfall (CVaR)
var_hist_95 = -np.percentile(port_returns, 5)
var_hist_99 = -np.percentile(port_returns, 1)
var_param_95 = -(port_returns.mean() - 1.64485 * port_returns.std())
tail_losses = port_returns[port_returns <= -var_hist_95]
cvar_95 = -tail_losses.mean() if not tail_losses.empty else var_hist_95

# 5. Beta & Alpha (vs SPY)
cov = np.cov(port_returns, bench_returns)[0, 1]
bench_var = np.var(bench_returns)
beta = cov / bench_var if bench_var > 0 else 1.0
bench_ann_ret = bench_returns.mean() * TRADING_DAYS
alpha = ann_ret - (RISK_FREE_RATE + beta * (bench_ann_ret - RISK_FREE_RATE))
track_err = (port_returns - bench_returns).std() * np.sqrt(TRADING_DAYS)
info_ratio = (ann_ret - bench_ann_ret) / track_err if track_err > 0 else 0.0

# 6. Euler Risk Attribution
asset_returns = held_prices.pct_change().dropna()
cov_matrix = asset_returns.cov() * TRADING_DAYS
sigma_p = np.sqrt(weights_arr.T @ cov_matrix.values @ weights_arr)
mcr = (cov_matrix.values @ weights_arr) / sigma_p
crc = weights_arr * mcr
pcr = crc / sigma_p

attribution_df = pd.DataFrame({
    "Symbol": held_symbols,
    "Weight": weights_arr,
    "Marginal Risk (MCR)": mcr,
    "Component Risk (CRC)": crc,
    "Risk Contribution (%)": pcr * 100,
})

print(f"=== QUANTITATIVE PERFORMANCE & RISK METRICS ===")
print(f"CAGR (Annualized):        {cagr * 100:+.2f}%")
print(f"Annualized Volatility:    {ann_vol * 100:.2f}%")
print(f"Sharpe Ratio:             {sharpe:.3f}")
print(f"Sortino Ratio:            {sortino:.3f}")
print(f"Maximum Drawdown:         {max_drawdown * 100:.2f}%")
print(f"Beta (vs SPY):            {beta:.3f}")
print(f"Jensen's Alpha:           {alpha * 100:+.2f}%")
print(f"95% Historical VaR (1d):  {var_hist_95 * 100:.2f}% (${var_hist_95 * total_mkt_val:,.2f})")
print(f"95% Expected Shortfall:   {cvar_95 * 100:.2f}% (${cvar_95 * total_mkt_val:,.2f})")
display(attribution_df)

## Step 6: Markowitz Modern Portfolio Theory (Optimization & Efficient Frontier)

In [ ]:
# Expected annual returns and covariance
mean_ann_returns = asset_returns.mean().values * TRADING_DAYS
cov_ann = cov_matrix.values
num_assets = len(held_symbols)

# Constraints: sum(w) = 1, 0 <= w_i <= 1
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
bounds = tuple((0.0, 1.0) for _ in range(num_assets))
init_weights = np.ones(num_assets) / num_assets

# 1. Minimum Variance Portfolio
def port_variance(w):
    return w.T @ cov_ann @ w

opt_min_var = optimize.minimize(port_variance, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
w_min_var = opt_min_var.x
ret_min_var = np.sum(w_min_var * mean_ann_returns)
vol_min_var = np.sqrt(w_min_var.T @ cov_ann @ w_min_var)
sharpe_min_var = (ret_min_var - RISK_FREE_RATE) / vol_min_var

# 2. Maximum Sharpe Ratio Portfolio
def neg_sharpe(w):
    r = np.sum(w * mean_ann_returns)
    v = np.sqrt(w.T @ cov_ann @ w)
    return -(r - RISK_FREE_RATE) / v if v > 0 else 1e6

opt_max_sharpe = optimize.minimize(neg_sharpe, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
w_max_sharpe = opt_max_sharpe.x
ret_max_sharpe = np.sum(w_max_sharpe * mean_ann_returns)
vol_max_sharpe = np.sqrt(w_max_sharpe.T @ cov_ann @ w_max_sharpe)
sharpe_max_sharpe = (ret_max_sharpe - RISK_FREE_RATE) / vol_max_sharpe

# 3. Simulate Efficient Frontier (3,000 portfolios)
np.random.seed(42)
sim_returns = []
sim_volatilities = []
sim_sharpes = []
for _ in range(3000):
    w_rand = np.random.dirichlet(np.ones(num_assets))
    r_sim = np.sum(w_rand * mean_ann_returns)
    v_sim = np.sqrt(w_rand.T @ cov_ann @ w_rand)
    s_sim = (r_sim - RISK_FREE_RATE) / v_sim if v_sim > 0 else 0
    sim_returns.append(r_sim)
    sim_volatilities.append(v_sim)
    sim_sharpes.append(s_sim)

print("=== MARKOWITZ OPTIMIZATION RESULTS ===")
print(f"Minimum Variance Allocation:")
print(f"  Return: {ret_min_var * 100:.2f}% | Volatility: {vol_min_var * 100:.2f}% | Sharpe: {sharpe_min_var:.3f}")
for s, w in zip(held_symbols, w_min_var):
    print(f"    {s:6s}: {w * 100:5.2f}%")

print(f"\nMaximum Sharpe Allocation:")
print(f"  Return: {ret_max_sharpe * 100:.2f}% | Volatility: {vol_max_sharpe * 100:.2f}% | Sharpe: {sharpe_max_sharpe:.3f}")
for s, w in zip(held_symbols, w_max_sharpe):
    print(f"    {s:6s}: {w * 100:5.2f}%")

## Step 7: Monte Carlo Correlated Simulation (5,000 Paths, 252 Days)

In [ ]:
# Multivariate Correlated Geometric Brownian Motion via Cholesky
num_sims = 5000
horizon = 252
np.random.seed(42)

cov_daily = (asset_returns.cov().to_numpy(copy=True)) + np.eye(num_assets) * 1e-6
L = np.linalg.cholesky(cov_daily)
mu_daily = asset_returns.mean().values
diag_cov = np.diag(cov_daily)

# Current prices
S0 = np.array([float(latest_prices[s]) for s in held_symbols])
qty_arr = np.array([PORTFOLIO_HOLDINGS[s]["qty"] for s in held_symbols])

sim_paths = np.zeros((horizon + 1, num_sims))
sim_paths[0, :] = total_mkt_val

current_prices = np.tile(S0, (num_sims, 1))  # (num_sims, num_assets)

for t in range(1, horizon + 1):
    z = np.random.standard_normal((num_sims, num_assets))
    correlated_normals = z @ L.T
    drift = (mu_daily - 0.5 * diag_cov)
    shocks = np.exp(drift + correlated_normals)
    current_prices = current_prices * shocks
    sim_paths[t, :] = current_prices @ qty_arr

terminal_values = sim_paths[-1, :]
p5 = np.percentile(terminal_values, 5)
p25 = np.percentile(terminal_values, 25)
p50 = np.percentile(terminal_values, 50)
p75 = np.percentile(terminal_values, 75)
p95 = np.percentile(terminal_values, 95)
loss_prob = np.mean(terminal_values < total_mkt_val) * 100

print(f"=== MONTE CARLO SIMULATION (5,000 PATHS, 1-YEAR) ===")
print(f"Initial Portfolio Value:      ${total_mkt_val:,.2f}")
print(f"Expected Mean Terminal Value: ${np.mean(terminal_values):,.2f}")
print(f"5th Percentile (Severe Bear): ${p5:,.2f} ({(p5 / total_mkt_val - 1) * 100:+.2f}%)")
print(f"25th Percentile (Weak Mkt):   ${p25:,.2f} ({(p25 / total_mkt_val - 1) * 100:+.2f}%)")
print(f"50th Percentile (Median):     ${p50:,.2f} ({(p50 / total_mkt_val - 1) * 100:+.2f}%)")
print(f"75th Percentile (Bull Mkt):   ${p75:,.2f} ({(p75 / total_mkt_val - 1) * 100:+.2f}%)")
print(f"95th Percentile (High Growth):${p95:,.2f} ({(p95 / total_mkt_val - 1) * 100:+.2f}%)")
print(f"Probability of Capital Loss:  {loss_prob:.1f}%")

## Step 8: Macroeconomic Scenario & Stress Testing

In [ ]:
SCENARIOS = {
    "2008 Global Financial Crash": {"Equity": -0.45, "Commodity": -0.30, "Fixed Income": +0.12, "REIT": -0.50},
    "2020 COVID Liquidity Shock": {"Equity": -0.30, "Commodity": -0.25, "Fixed Income": +0.06, "REIT": -0.35},
    "2022 Tech Stagflation":      {"Equity": -0.25, "Commodity": +0.20, "Fixed Income": -0.15, "REIT": -0.20},
    "Tech & Risk-On Meltup":      {"Equity": +0.30, "Commodity": +0.10, "Fixed Income": -0.04, "REIT": +0.15},
}

stress_results = []
for sc_name, shocks in SCENARIOS.items():
    stressed_total = 0.0
    for _, row in holdings_df.iterrows():
        asset_cls = row["class"]
        shock = shocks.get(asset_cls, 0.0)
        stressed_val = row["market_value"] * (1 + shock)
        stressed_total += stressed_val
    impact_dollar = stressed_total - total_mkt_val
    impact_pct = (impact_dollar / total_mkt_val) * 100
    stress_results.append({
        "Scenario": sc_name,
        "Stressed Value": stressed_total,
        "Dollar Impact": impact_dollar,
        "Percentage Impact (%)": impact_pct,
    })

stress_df = pd.DataFrame(stress_results)
print("=== MACROECONOMIC STRESS TESTING RESULTS ===")
display(stress_df)

## Step 9: Interactive Plotly Visualizations Dashboard

In [ ]:
# 1. Cumulative Performance vs SPY
fig_perf = go.Figure()
norm_port = (port_daily_value / port_daily_value.iloc[0]) * 100
norm_spy = (price_matrix["SPY"].loc[port_daily_value.index] / price_matrix["SPY"].loc[port_daily_value.index].iloc[0]) * 100

fig_perf.add_trace(go.Scatter(x=norm_port.index, y=norm_port, name="My Portfolio", line=dict(color="#2563EB", width=2.5)))
fig_perf.add_trace(go.Scatter(x=norm_spy.index, y=norm_spy, name="SPY Benchmark", line=dict(color="#64748B", dash="dash")))
fig_perf.update_layout(
    title="Cumulative Performance (Normalized to 100)",
    xaxis_title="Date",
    yaxis_title="Growth Factor",
    template="plotly_white",
    height=450,
)
fig_perf.show()

# 2. Markowitz Efficient Frontier Chart
fig_ef = go.Figure()
fig_ef.add_trace(go.Scatter(
    x=np.array(sim_volatilities) * 100,
    y=np.array(sim_returns) * 100,
    mode="markers",
    marker=dict(size=4, color=sim_sharpes, colorscale="Viridis", showscale=True, colorbar=dict(title="Sharpe")),
    name="Simulated Portfolios",
))
fig_ef.add_trace(go.Scatter(
    x=[vol_min_var * 100], y=[ret_min_var * 100],
    mode="markers+text", name="Minimum Variance",
    marker=dict(color="blue", size=14, symbol="diamond"),
    text=["Min Volatility"], textposition="top center",
))
fig_ef.add_trace(go.Scatter(
    x=[vol_max_sharpe * 100], y=[ret_max_sharpe * 100],
    mode="markers+text", name="Maximum Sharpe",
    marker=dict(color="red", size=14, symbol="star"),
    text=["Max Sharpe"], textposition="top center",
))
fig_ef.add_trace(go.Scatter(
    x=[ann_vol * 100], y=[ann_ret * 100],
    mode="markers+text", name="Current Allocation",
    marker=dict(color="black", size=12, symbol="circle"),
    text=["Current"], textposition="bottom center",
))
fig_ef.update_layout(
    title="Markowitz Efficient Frontier & Asset Allocation",
    xaxis_title="Annualized Volatility (%)",
    yaxis_title="Expected Annual Return (%)",
    template="plotly_white",
    height=500,
)
fig_ef.show()

# 3. Monte Carlo Simulation Fan Chart
fig_mc = go.Figure()
days = np.arange(horizon + 1)
# Plot first 50 sample paths in light grey
for i in range(50):
    fig_mc.add_trace(go.Scatter(x=days, y=sim_paths[:, i], mode="lines", line=dict(color="rgba(150, 150, 150, 0.15)"), showlegend=False))

p5_line = np.percentile(sim_paths, 5, axis=1)
p50_line = np.percentile(sim_paths, 50, axis=1)
p95_line = np.percentile(sim_paths, 95, axis=1)

fig_mc.add_trace(go.Scatter(x=days, y=p95_line, mode="lines", name="95th Percentile (Bull)", line=dict(color="green", width=2.5)))
fig_mc.add_trace(go.Scatter(x=days, y=p50_line, mode="lines", name="50th Percentile (Median)", line=dict(color="blue", width=3)))
fig_mc.add_trace(go.Scatter(x=days, y=p5_line, mode="lines", name="5th Percentile (Bear)", line=dict(color="red", width=2.5)))
fig_mc.update_layout(
    title="Monte Carlo 1-Year Projected Portfolio Wealth Distribution (5,000 Paths)",
    xaxis_title="Trading Days Horizon",
    yaxis_title="Portfolio Value ($)",
    template="plotly_white",
    height=500,
)
fig_mc.show()

# 4. Macroeconomic Stress Testing Bar Chart
colors = ['#EF4444' if x < 0 else '#10B981' for x in stress_df["Percentage Impact (%)"]]
fig_stress = go.Figure(go.Bar(
    x=stress_df["Scenario"],
    y=stress_df["Percentage Impact (%)"],
    marker_color=colors,
    text=[f"{x:+.2f}%" for x in stress_df["Percentage Impact (%)"]],
    textposition="outside",
))
fig_stress.update_layout(
    title="Portfolio Impact Under Macroeconomic Stress Scenarios",
    xaxis_title="Historical Stress Scenario",
    yaxis_title="Value Impact (%)",
    template="plotly_white",
    height=450,
)
fig_stress.show()